# Aula 07 – Validade de Argumentos e Regras de Inferência

## 1. Fundamentação Teórica
Validação matemática de argumentos dedutivos, aplicação de Modus Ponens e Modus Tollens para diagnóstico de falhas, e motor de inferência em tempo real para telemetria SCADA.

In [ ]:
import itertools

def check_argument_validity(premises, conclusion, var_names):
    is_valid = True
    print(f"{' | '.join(var_names)} | P1 | P2 | P3 | Concl | Valido")
    print("-" * 50)
    for comb in itertools.product([False, True], repeat=len(var_names)):
        env = {var_names[i]: comb[i] for i in range(len(var_names))}
        p_vals = [p(env) for p in premises]
        all_p = all(p_vals)
        c_val = conclusion(env)
        
        if all_p and not c_val:
            is_valid = False
            
        env_str = " | ".join(str(int(comb[i])) for i in range(len(var_names)))
        p_str = " |  ".join(str(int(v)) for v in p_vals)
        print(f" {env_str} |  {p_str} |   {int(c_val)}   |  {all_p <= c_val}")
    return is_valid

# Variáveis do processo: Corrente Alta (C), Temperatura Alta (T), Alarme BMS (A)
vars_diag = ['C', 'T', 'A']
prem1 = lambda e: (not e['C']) or e['T']
prem2 = lambda e: (not e['T']) or e['A']
prem3 = lambda e: not e['A']
concl = lambda e: not e['C']

valid = check_argument_validity([prem1, prem2, prem3], concl, vars_diag)
print("-" * 50)
print(f"O argumento dedutivo de diagnóstico é VÁLIDO? -> {'SIM (100% CORRETO)' if valid else 'NAO'}\n")


## 2. Prova Computacional: Falácia da Afirmação do Consequente

**Enunciado:**
Prove computacionalmente que o argumento da Falácia da Afirmação do Consequente ($P_1: A \to B, P_2: B \vdash A$) é **Inválido**, exibindo a linha da tabela de verdade que constitui o contraexemplo.

In [ ]:
# Resolução do Exercício - Falácia da Afirmação do Consequente
contraexemplos = []
for A, B in itertools.product([False, True], repeat=2):
    p1 = (not A) or B
    p2 = B
    conclusao = A
    if p1 and p2 and not conclusao:
        contraexemplos.append((A, B))

print("--- EXERCÍCIO DA FALÁCIA ---")
print(f"Total de contraexemplos: {len(contraexemplos)}")
print(f"Contraexemplo: A={contraexemplos[0][0]}, B={contraexemplos[0][1]}")
print("Conclusão: Argumento INVÁLIDO (Falácia da Afirmação do Consequente).\n")


## 3. Motor de Diagnóstico SCADA em Tempo Real

Aplicação prática das regras de inferência em pacotes de telemetria recebidos da estação de voo para prevenção de inundação de alarmes e isolamento de falhas.

In [ ]:
# ========================================================
# SIMULADOR DE INFERÊNCIA SCADA (NOVO RECURSO)
# ========================================================
def scada_diagnostic_engine(telemetry_data: dict):
    print(f"--- Processando Pacote de Telemetria: {telemetry_data} ---")
    
    i_battery = telemetry_data.get("i_battery", 0.0)
    t_bms = telemetry_data.get("t_bms", 0.0)
    alarm_bms = telemetry_data.get("alarm_bms", False)
    
    c_excessiva = i_battery > 60.0
    t_excessiva = t_bms > 60.0
    
    if not alarm_bms:
        print("[INFERÊNCIA] Alarme geral inativo. Pela regra de Modus Tollens: ¬A -> ¬C.")
        print("-> STATUS: Corrente de descarga garantida em faixa segura.")
        
        if t_excessiva:
            print("-> DIAGNÓSTICO IHM: Leitura térmica elevada detectada sem confirmação pelo sistema de alarme.")
            print("-> AÇÃO RECOMENDADA: Alerta de calibração/falha de leitura no termopar do BMS.\n")
        else:
            print("-> STATUS: Subsistema de potência operando em conformidade térmica e elétrica.\n")
    else:
        print("[CRÍTICO] Alarme BMS Ativo! Procedimento de retorno seguro (RTL) disparado.\n")

# Testando com a telemetria do drone
scada_diagnostic_engine({"i_battery": 42.5, "t_bms": 38.0, "alarm_bms": False})
scada_diagnostic_engine({"i_battery": 35.0, "t_bms": 68.5, "alarm_bms": False})
scada_diagnostic_engine({"i_battery": 75.0, "t_bms": 72.0, "alarm_bms": True})
